# บทที่ 2 — สอนคอมพิวเตอร์ให้ "แยกสี" หาตุ๊กตา

🎯 เป้าหมาย: เข้าใจว่าทำไมต้องแปลงภาพเป็น HSV ก่อนแยกสี (ไม่ใช้ BGR ตรงๆ) แล้วลงมือแยกสีจริงจากภาพ

## 1. ปัญหาของ BGR/RGB: แสงเปลี่ยน สีก็เปลี่ยนตาม

ถ้าเราแยกวัตถุด้วยค่า B, G, R ตรงๆ จะมีปัญหา: **ใบไม้สีเขียวในที่มืด** กับ **ใบไม้สีเขียวในที่สว่างจ้า**
ค่า B, G, R ของมันต่างกันเยอะมาก (แสงน้อย = ทุกค่าต่ำลงหมด) ทั้งที่ "เป็นสีเขียวเหมือนกัน" ในสายตาคน

**HSV** แก้ปัญหานี้โดยแยก "เนื้อสี" ออกจาก "ความสว่าง":

| ตัวย่อ | ชื่อเต็ม | ความหมาย | ช่วงค่าใน OpenCV |
|---|---|---|---|
| **H** | Hue | เนื้อสี/โทนสี (แดง เขียว น้ำเงิน ...) | 0-179 |
| **S** | Saturation | ความสดของสี (0=เทาซีด, สูง=สดจัด) | 0-255 |
| **V** | Value | ความสว่าง (0=มืด/ดำ, สูง=สว่าง) | 0-255 |

แสงเปลี่ยน → **V เปลี่ยน แต่ H แทบไม่เปลี่ยน** — นี่คือเหตุผลที่ vision งานจริงแทบทุกที่แยกสีด้วย HSV ไม่ใช่ RGB

## 2. ลองดู "แถบสี Hue" จริงๆ ให้เห็นภาพ

สร้างแถบไล่ค่า H ตั้งแต่ 0-179 (คงค่า S=V=255 ไว้ให้สดและสว่างที่สุด) แล้วแปลงกลับเป็นภาพให้ดู

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Leelawadee UI"   # ฟอนต์นี้รองรับภาษาไทย กันตัวอักษรไทยในกราฟกลายเป็นกล่องว่าง

h_values = np.arange(180)                                  # ตัวเลข 0,1,2,...,179
hue_bar_hsv = np.zeros((60, 180, 3), np.uint8)
hue_bar_hsv[:, :, 0] = h_values                             # ช่อง H = ไล่เลขตามคอลัมน์
hue_bar_hsv[:, :, 1] = 255                                  # S เต็ม (สดสุด)
hue_bar_hsv[:, :, 2] = 255                                  # V เต็ม (สว่างสุด)

hue_bar_bgr = cv2.cvtColor(hue_bar_hsv, cv2.COLOR_HSV2BGR)  # แปลง HSV -> BGR เพื่อโชว์เป็นสีจริง

plt.figure(figsize=(8, 2))
plt.imshow(cv2.cvtColor(hue_bar_bgr, cv2.COLOR_BGR2RGB))
plt.xlabel("ค่า H (0-179)")
plt.yticks([])
plt.title("แถบสี Hue ทั้งหมดที่เป็นไปได้ (S=V=255)")
plt.show()

print("จำง่ายๆ:  H≈0 หรือ 179 = แดง  |  H≈30 = เหลือง  |  H≈60 = เขียว  |  H≈120 = น้ำเงิน")

สังเกตว่า H=0 กับ H=179 ให้สีแดงเหมือนกัน (วงล้อสีมันวนกลับมาบรรจบ) — เดี๋ยวมีผลตอนจะแยก "สีแดง" ทีหลัง (เป็น edge case ที่ต้องระวัง)

## 3. แปลงภาพเป็น HSV แล้วดูค่าตัวเลข

บทนี้ใช้ **"ภาพตัวอย่างสนาม"** ที่สร้างขึ้นตายตัว (มีตุ๊กตา 3 ตัวเหมือนโปรเจคจริง: เขียว/น้ำตาล/เทา)
แทนที่จะถ่ายจากกล้องสด เพื่อให้ตัวเลขในบทเรียนตรงกับที่อธิบายไว้เป๊ะทุกครั้งที่รัน (กล้องจริงแสง/มุมไม่แน่นอน ตัวเลขจะขยับ)
— พอจบบทจะมีเซลล์โบนัสให้ลองกับกล้องจริงของตัวเองด้วย

In [ ]:
def ภาพตัวอย่างสนาม():
    # สร้างภาพจำลอง "สนามยิง" มีตุ๊กตา 3 ตัวเหมือนโปรเจคจริง — ตำแหน่ง/สีตายตัวทุกครั้งที่เรียก
    img = np.full((480, 640, 3), (235, 230, 220), np.uint8)          # พื้นหลังห้อง
    cv2.rectangle(img, (0, 340), (640, 480), (170, 140, 90), -1)     # โต๊ะวางตุ๊กตา
    for x, y, r, color in [(160, 300, 55, (40, 170, 40)),    # เขียว  = ไดโนเสาร์
                           (330, 290, 40, (60, 110, 150)),   # น้ำตาล = คาปิบาร่า
                           (480, 280, 28, (130, 130, 130))]:  # เทา    = ช้าง
        cv2.circle(img, (x, y), r, color, -1)
    return img


frame = ภาพตัวอย่างสนาม()
hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)   # แปลงทั้งภาพจาก BGR เป็น HSV ทีเดียว

# ดูค่า HSV ตรงตำแหน่ง (300, 160) ซึ่งอยู่ในตัวไดโนเสาร์เขียวพอดี
print("ค่า HSV ที่พิกัด (y=300, x=160):", hsv[300, 160])
print("(เทียบ H≈40-80 ก็คือโซนสีเขียวตามแถบ Hue ด้านบนพอดี)")

## 4. `cv2.inRange` — สร้าง "หน้ากาก" (mask) ของสีที่ต้องการ

`cv2.inRange(hsv, ล่าง, บน)` เช็คทุกพิกเซลว่าค่าอยู่ในช่วงที่กำหนดไหม → คืนภาพขาวดำ (mask):
**ขาว (255) = อยู่ในช่วงสี, ดำ (0) = ไม่อยู่ในช่วง**

In [ ]:
เขียวล่าง = np.array([35, 80, 60])     # [H, S, V] ขอบล่างของสีเขียว
เขียวบน = np.array([85, 255, 255])     # [H, S, V] ขอบบนของสีเขียว

mask = cv2.inRange(hsv, เขียวล่าง, เขียวบน)

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); ax[0].set_title("ภาพต้นฉบับ")
ax[1].imshow(mask, cmap="gray"); ax[1].set_title("mask (ขาว = พบสีเขียว)")
result = cv2.bitwise_and(frame, frame, mask=mask)   # โชว์เฉพาะส่วนที่ mask เป็นขาว
ax[2].imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB)); ax[2].set_title("ผลลัพธ์ (เฉพาะส่วนเขียว)")
plt.show()

print("จำนวนพิกเซลที่ถูกจับว่าเป็นสีเขียว:", np.count_nonzero(mask))

## 5. ล้าง noise ด้วย Morphology

ในสถานการณ์จริง แสง/เงาทำให้ mask มีจุดขาวเล็กๆ กระจายทั่วภาพ (noise) ที่ไม่ใช่วัตถุจริง
เราจำลองสถานการณ์นี้โดยเติม "noise" ลงใน mask เอง แล้วดูวิธีล้างมันออก

In [ ]:
np.random.seed(0)   # ล็อกค่าสุ่มไว้ ให้ผลลัพธ์เหมือนเดิมทุกครั้งที่รัน (เพื่อการเรียนที่ทำนายผลได้)

mask_noisy = mask.copy()
ys = np.random.randint(0, mask.shape[0], 300)
xs = np.random.randint(0, mask.shape[1], 300)
mask_noisy[ys, xs] = 255   # โรยจุดขาวสุ่ม 300 จุดทั่วภาพ จำลอง noise

kernel = np.ones((5, 5), np.uint8)
mask_opened = cv2.morphologyEx(mask_noisy, cv2.MORPH_OPEN, kernel)    # OPEN = กัดจุดขาวเล็กๆ ทิ้ง
mask_closed = cv2.morphologyEx(mask_opened, cv2.MORPH_CLOSE, kernel)  # CLOSE = อุดรูดำเล็กๆ ในก้อนขาว

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(mask_noisy, cmap="gray"); ax[0].set_title("mask ที่มี noise")
ax[1].imshow(mask_opened, cmap="gray"); ax[1].set_title("หลัง OPEN (noise หายไปมาก)")
ax[2].imshow(mask_closed, cmap="gray"); ax[2].set_title("หลัง CLOSE (ก้อนวัตถุทึบขึ้น)")
plt.show()

## 6. 🎛️ อินเทอร์แอคทีฟ: จูนช่วงสีด้วยแถบเลื่อน

ลองใช้ `ipywidgets` สร้างแถบเลื่อนในสมุดบันทึกเลย เลื่อนแล้วภาพขวาจะเปลี่ยนทันที (ต้องรันในสมุดบันทึกจริงถึงจะลากได้ ในการรันแบบทดสอบอัตโนมัตินี้จะโชว์แค่ค่าเริ่มต้น)

In [ ]:
from ipywidgets import interact, IntSlider

def จูนสี(h_min=35, h_max=85, s_min=80, v_min=60):
    lower = np.array([h_min, s_min, v_min])
    upper = np.array([h_max, 255, 255])
    m = cv2.inRange(hsv, lower, upper)
    plt.figure(figsize=(5, 4))
    plt.imshow(m, cmap="gray")
    plt.title(f"H:[{h_min}-{h_max}] S_min:{s_min} V_min:{v_min}  (พิกเซลขาว: {np.count_nonzero(m)})")
    plt.show()

interact(จูนสี,
         h_min=IntSlider(35, 0, 179), h_max=IntSlider(85, 0, 179),
         s_min=IntSlider(80, 0, 255), v_min=IntSlider(60, 0, 255));

## 🧪 แบบฝึกหัดท้ายบท

โจทย์: ในภาพจำลองมีตุ๊กตา**สีน้ำตาล** (คาปิบาร่า) อยู่ด้วย ลองหาช่วง HSV ที่จับเฉพาะสีน้ำตาลได้
(ใช้แถบเลื่อนด้านบนลองไล่ค่าดู หรือเขียนโค้ดล่างนี้ปรับค่าเอง)

ใบ้: จากแถบ Hue บนสุดของบทนี้ สีน้ำตาล/ส้มอยู่แถว H ≈ 10-25

In [ ]:
# ปรับค่าตรงนี้แล้วรันดู
น้ำตาลล่าง = np.array([10, 60, 60])
น้ำตาลบน = np.array([25, 255, 255])

mask_brown = cv2.inRange(hsv, น้ำตาลล่าง, น้ำตาลบน)
plt.imshow(mask_brown, cmap="gray")
plt.title(f"พิกเซลขาว: {np.count_nonzero(mask_brown)}")
plt.show()

<details><summary>👉 คลิกดูเฉลย (ค่าที่ใช้ได้จริงในโปรเจค อยู่ใน src/config.py ด้วย)</summary>

```python
น้ำตาลล่าง = np.array([10, 60, 60])
น้ำตาลบน = np.array([25, 255, 255])
```

ถ้าพิกเซลขาวเป็น 0 หรือน้อยผิดปกติ แปลว่าช่วงแคบไป/ผิดโซน ลองขยาย S_min, V_min ให้ต่ำลง
</details>

## 🎥 โบนัส: ลองกับกล้องจริงของตัวเอง

หยิบของสีสด (ขวดน้ำ ของเล่น ผลไม้) มาส่องกล้อง แล้วลองจูนช่วง HSV ของตัวเองดู
(ถ้าไม่มีกล้อง เซลล์นี้จะใช้ภาพตัวอย่างสนามซ้ำ ไม่ error)

In [ ]:
def _ถ่ายจริงถ้ามี(index=0):
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW)
    if cap.isOpened():
        ok, cam_frame = cap.read()
        cap.release()
        if ok:
            print(f"✅ ถ่ายจากกล้องจริงสำเร็จ! (index {index})")
            return cam_frame
    print("⚠️ ไม่พบกล้อง — ใช้ภาพตัวอย่างสนามซ้ำแทน")
    return ภาพตัวอย่างสนาม()


my_frame = _ถ่ายจริงถ้ามี()
my_hsv = cv2.cvtColor(my_frame, cv2.COLOR_BGR2HSV)

# ปรับ 6 ค่านี้ให้เข้ากับสีของที่ถืออยู่ (ใช้ hue bar ข้อ 2 เป็นตัวช่วยกะ H)
my_mask = cv2.inRange(my_hsv, np.array([35, 80, 60]), np.array([85, 255, 255]))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(cv2.cvtColor(my_frame, cv2.COLOR_BGR2RGB)); ax[0].set_title("ภาพของคุณ")
ax[1].imshow(my_mask, cmap="gray"); ax[1].set_title(f"mask (ขาว: {np.count_nonzero(my_mask)} px)")
plt.show()

---
## ✅ สรุปบทนี้

| เรื่อง | สรุปสั้นๆ |
|---|---|
| ทำไมใช้ HSV | แยก "เนื้อสี (H)" ออกจาก "ความสว่าง (V)" ทนแสงเปลี่ยนกว่า BGR |
| `cv2.cvtColor(img, cv2.COLOR_BGR2HSV)` | แปลงทั้งภาพเป็น HSV |
| `cv2.inRange(hsv, lower, upper)` | สร้าง mask ขาว-ดำจากช่วงสี |
| Morphology OPEN/CLOSE | ล้าง noise เล็กๆ ออกจาก mask |

➡️ **ไปต่อ:** เปิด `03_tracking_and_aiming.ipynb` — เอา mask นี้ไปหา "ตำแหน่ง" ของวัตถุ แล้วคำนวณว่าป้อมต้องหมุนไปทางไหน

🎛️ **อยากจูนด้วยแถบเลื่อนแบบลากได้เต็มจอ (ภาพสด)?** ลองรัน `learn/02_hsv_tuner.py`